In [ ]:
%pip install pdfplumber

In [7]:
import os
import pdfplumber
from pathlib import Path

DATA_DIR = Path("data")

def load_documents():
    docs = []
    for path in DATA_DIR.glob("*"):
        if path.suffix == ".pdf":
            text = ""
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    text += page.extract_text() + "\n"
            docs.append({"filename": path.name, "text": text})

    return docs

docs = load_documents()
len(docs), docs[0]["filename"]


(1, '68-3353-01_D_sharelink_pro_1000.pdf')

In [8]:
def chunk_text(text, max_chars=800):
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk)
        start = end
    return chunks

chunks = []
for doc in docs:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "source": doc["filename"],
            "text": chunk
        })

len(chunks), chunks[0]

(136,
 {'source': '68-3353-01_D_sharelink_pro_1000.pdf',
  'text': 'User Guide\nCollaboration Systems\nShareLink Pro 1000\nWireless and Wired Collaboration Gateway\n68-3353-01 Rev. D\n11 20\nSafety Instructions\nSafety Instructions • English Istruzioni di sicurezza • Italiano\nWARNING: This symbol, , when used on the product, is intended to AVVERTENZA: Il simbolo, , se usato sul prodotto, serve ad\nalert the user of the presence of uninsulated dangerous voltage within the avvertire l’utente della presenza di tensione non isolata pericolosa\nproduct’s enclosure that may present a risk of electric shock. all’interno del contenitore del prodotto che può costituire un rischio di\nscosse elettriche.\nATTENTION: This symbol, , when used on the product, is intended\nto alert the user of important operating and maintenance (servicing) ATTENTZIONE: Il simbolo, , se usato sul pr'})

In [ ]:
%pip install sentence_transformers

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast model

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = model.encode(chunk_texts, convert_to_numpy=True)
chunk_embeddings.shape

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1732.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(136, 384)

In [10]:
def retrieve_relevant_chunks(query, top_k=5):
    query_emb = model.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(query_emb, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_idx:
        results.append({
            "score": float(sims[idx]),
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"]
        })
    return results

results = retrieve_relevant_chunks("What is the sharelink used for?")
results[0]

{'score': 0.5561026930809021,
 'source': '68-3353-01_D_sharelink_pro_1000.pdf',
 'text': 'es.\nFigure 1. ShareLink Pro 1000 Application Diagram\nShareLink Pro 1000 • Introduction 1\nFeatures\n• Wirelessly share content from mobile devices — Connects a wide variety of\ndevices to the system wirelessly or through a wired LAN connection.\n• Provides full screen mirroring for all devices — Display the entire screen of your\ndevice through the wireless collaboration gateway for more fluid and easy collaboration\nsessions.\n• Wireless and wired sources can collaborate simultaneously — HDMI input\nenables wired users or sources to collaborate simultaneously with wireless users in the\nsame session.\n• Supports Mac and Windows computers as well as Apple and Android\ntablets and smartphones\n• Dedicated app provides consistent user experience across platforms —\nSimilar interface for all platforms mak'}

In [11]:
def answer_question(query, top_k=3):
    results = retrieve_relevant_chunks(query, top_k=top_k)
    print(f"Question: {query}\n")
    print("Most relevant information:\n")
    for r in results:
        print(f"Source: {r['source']} (score: {r['score']:.3f})")
        print(r["text"])
        print("-" * 80)

answer_question("What are the main features of the sharelink?")

Question: What are the main features of the sharelink?

Most relevant information:

Source: 68-3353-01_D_sharelink_pro_1000.pdf (score: 0.527)
es.
Figure 1. ShareLink Pro 1000 Application Diagram
ShareLink Pro 1000 • Introduction 1
Features
• Wirelessly share content from mobile devices — Connects a wide variety of
devices to the system wirelessly or through a wired LAN connection.
• Provides full screen mirroring for all devices — Display the entire screen of your
device through the wireless collaboration gateway for more fluid and easy collaboration
sessions.
• Wireless and wired sources can collaborate simultaneously — HDMI input
enables wired users or sources to collaborate simultaneously with wireless users in the
same session.
• Supports Mac and Windows computers as well as Apple and Android
tablets and smartphones
• Dedicated app provides consistent user experience across platforms —
Similar interface for all platforms mak
--------------------------------------------------------

In [12]:
def build_context(results):
    context = ""
    for r in results:
        context += f"From {r['source']}:\n{r['text']}\n\n"
    return context

def build_prompt(query, results):
    context = build_context(results)
    prompt = f"""You are an assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not there, say you don't know.

Context:
{context}

Question: {query}
Answer:"""
    return prompt

### Fully Local Model

In [ ]:
%pip install bitsandbytes

In [ ]:
%pip install torch

In [ ]:
%pip install accelerate

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(llm_name)
model_llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float32,
    device_map="cpu"
)
# model_llm = AutoModelForCausalLM.from_pretrained(llm_name, torch_dtype=torch.float32)

Loading weights: 100%|██████████| 453/453 [00:24<00:00, 18.20it/s]


In [14]:
def build_prompt(query, retrieved_chunks):
    context = ""
    for r in retrieved_chunks:
        context += f"Source: {r['source']}\n{r['text']}\n\n"

    prompt = f"""
You are a helpful assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:
"""
    return prompt

In [15]:
import torch

def generate_answer(prompt, max_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer[len(prompt):].strip()

In [16]:
def ask_extron(query, top_k=2):
    retrieved = retrieve_relevant_chunks(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    answer = generate_answer(prompt)
    return answer
    
    # print("QUESTION:")
    # print(query)
    # print("\nANSWER:")
    # print(answer)
    
    # print("\n--- Retrieved Chunks Used ---")
    # for r in retrieved:
    #     print(f"{r['source']} (score {r['score']:.3f})")

In [17]:
ask_extron("What are the main features of the sharelink?")

'atures of the sharelink are:\n\n- It wirelessly shares content from mobile devices.\n- It provides full screen mirroring for all devices.\n- It allows wireless and wired sources to collaborate simultaneously.\n- It supports Mac and Windows computers as well as Apple and Android tablets and smartphones.\n- It has a dedicated app that provides a consistent user experience across platforms.'

### Chat History

In [18]:
chat_history = []

In [19]:
def format_chat_history(history):
    text = ""
    for turn in history:
        text += f"User: {turn['user']}\n"
        text += f"Assistant: {turn['assistant']}\n\n"
    return text

In [20]:
def build_chat_prompt(query, retrieved_chunks, history):
    context = ""
    for r in retrieved_chunks:
        context += f"Source: {r['source']}\n{r['text']}\n\n"

    history_text = format_chat_history(history)

    prompt = f"""
You are a helpful assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not in the context, say you don't know.

Conversation so far:
{history_text}

Context:
{context}

User: {query}
Assistant:
"""
    return prompt

In [21]:
import torch

def generate_answer(prompt, max_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    # Decode only the newly generated text
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_output[len(prompt):].strip()
    return answer

In [22]:
def chat(query, top_k=3):
    # Retrieve relevant chunks
    retrieved = retrieve_relevant_chunks(query, top_k=top_k)

    # Build prompt with history
    prompt = build_chat_prompt(query, retrieved, chat_history)

    # Generate answer
    answer = generate_answer(prompt)

    # Save to history
    chat_history.append({
        "user": query,
        "assistant": answer
    })

    # Display
    print(f"User: {query}\n")
    print(f"Assistant: {answer}\n")
    print("--- Retrieved Chunks Used ---")
    for r in retrieved:
        print(f"{r['source']} (score {r['score']:.3f})")

In [ ]:
chat("What does the sharelink do?")

In [ ]:
%pip install gradio

In [23]:
import gradio as gr

def chat_fn(query):
    answer = ask_extron(query)
    return answer

ui = gr.Interface(
    fn=chat_fn,
    inputs=gr.Textbox(label="Ask about Extron products"),
    outputs=gr.Textbox(label="Answer"),
    title="Extron QA Assistant",
    description="Ask any question about Extron products. Powered by local RAG + LLM.",
)

ui.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
